# **Proyek ETL Pipeline Bank Marketing (Databricks Free Edition)**
## **Layer 1: Bronze Layer — Incremental File-Based Ingestion (Task 1 Workflow)**

Notebook ini berperan sebagai **Task 1 (`Ingest_Bronze`)** dalam Databricks Lakeflow Job/Workflow.

### **Desain & Jaminan Kualitas Layer Bronze:**
1. **Incremental Berbasis File (`source_file`):** Hanya membaca file batch CSV di Unity Catalog Volume `/Volumes/<catalog>/<schema>/<volume_name>/incoming/` yang **belum pernah tercatat** pada tabel Delta Bronze.
2. **Idempotent & Aman Dijalankan Ulang:** Tidak pernah menghapus atau me-reset data historis. Jika file yang sama diproses ulang (misal via parameter `target_file`), sistem otomatis melewati proses penulisan (`0` baris ditambahkan) sehingga **tidak terjadi duplikasi**.
3. **Skema Eksplisit (`StructType`):** Mencegah kegagalan *schema inference/mismatch* pada batch awal (misalnya ketika 150 baris pertama seluruhnya bernilai `"unknown"` pada kolom `contact` atau `poutcome`).
4. **Metadata Audit Lengkap:** Menyematkan `batch_id`, `source_file`, `source_path`, dan `ingested_at` lalu menyimpan sebagai tabel **Managed Delta** `{catalog}.{schema}.{bronze_table}`.

---
### **1. Parameter Task Workflow (`dbutils.widgets`)**

In [ ]:
import json
import os
from datetime import datetime, timezone
from pyspark.sql import functions as F
from pyspark.sql import types as T

# Parameter Job / Workflow (dapat diisi melalui Task Parameters di Databricks Jobs)
dbutils.widgets.text("catalog", "workspace", "Unity Catalog Name")
dbutils.widgets.text("schema", "bank_marketing", "Schema Name")
dbutils.widgets.text("volume_name", "raw_data", "Volume Name")
dbutils.widgets.text("bronze_table", "bronze_bank_marketing", "Nama Tabel Delta Bronze")
dbutils.widgets.dropdown(
    "ingest_mode",
    "next_one",
    ["next_one", "all_new"],
    "Mode Ingest (next_one / all_new)",
)
dbutils.widgets.text(
    "target_file",
    "",
    "Opsional: Nama File Spesifik (contoh: bank_batch_2008.csv untuk uji idempotency)",
)

catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()
volume_name = dbutils.widgets.get("volume_name").strip()
bronze_table = dbutils.widgets.get("bronze_table").strip()
ingest_mode = dbutils.widgets.get("ingest_mode").strip()
target_file = dbutils.widgets.get("target_file").strip()

if ingest_mode not in ("next_one", "all_new"):
    raise ValueError(f"Parameter ingest_mode tidak valid: '{ingest_mode}'. Gunakan 'next_one' atau 'all_new'.")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")

incoming_dir = f"/Volumes/{catalog}/{schema}/{volume_name}/incoming"
full_bronze_table = f"{catalog}.{schema}.{bronze_table}"

if not os.path.isdir(incoming_dir):
    raise FileNotFoundError(
        f"Direktori incoming tidak ditemukan di {incoming_dir}. "
        "Jalankan notebook 00_EDA_dan_Akuisisi.ipynb terlebih dahulu."
    )

print(f"Incoming Directory : {incoming_dir}")
print(f"Target Delta Table : {full_bronze_table}")
print(f"Mode Ingest        : {ingest_mode}")
print(f"Target File        : {target_file if target_file else '(Otomatis file baru)'}")

---
### **2. Definisi Skema Eksplisit Data Mentah (`StructType`)**

In [ ]:
BRONZE_RAW_SCHEMA = T.StructType([
    T.StructField("age", T.IntegerType(), True),
    T.StructField("job", T.StringType(), True),
    T.StructField("marital", T.StringType(), True),
    T.StructField("education", T.StringType(), True),
    T.StructField("default", T.StringType(), True),
    T.StructField("balance", T.IntegerType(), True),
    T.StructField("housing", T.StringType(), True),
    T.StructField("loan", T.StringType(), True),
    T.StructField("contact", T.StringType(), True),
    T.StructField("day", T.IntegerType(), True),
    T.StructField("month", T.StringType(), True),
    T.StructField("duration", T.IntegerType(), True),
    T.StructField("campaign", T.IntegerType(), True),
    T.StructField("pdays", T.IntegerType(), True),
    T.StructField("previous", T.IntegerType(), True),
    T.StructField("poutcome", T.StringType(), True),
    T.StructField("y", T.StringType(), True),
])

print(f"Skema eksplisit siap: {len(BRONZE_RAW_SCHEMA.fields)} kolom asli.")

---
### **3. Deteksi File Baru & Penjaga Idempotency**

In [ ]:
incoming_files = sorted(
    f for f in os.listdir(incoming_dir) if f.lower().endswith(".csv")
)
if not incoming_files:
    raise FileNotFoundError(
        f"Tidak ada file CSV di {incoming_dir}. Jalankan 00_EDA_dan_Akuisisi.ipynb terlebih dahulu."
    )

table_exists = spark.catalog.tableExists(full_bronze_table)
if table_exists:
    existing_df = spark.table(full_bronze_table)
    rows_before = existing_df.count()
    ingested_files = {
        row["source_file"]
        for row in existing_df.select("source_file").distinct().collect()
    }
else:
    rows_before = 0
    ingested_files = set()

files_to_ingest = []
if target_file:
    if target_file not in incoming_files:
        raise FileNotFoundError(
            f"File target_file '{target_file}' tidak ditemukan di {incoming_dir}. "
            f"File tersedia: {incoming_files}"
        )
    if target_file in ingested_files:
        print(f"[IDEMPOTENT SKIP] File '{target_file}' sudah ada di {full_bronze_table}. 0 baris ditambahkan.")
    else:
        files_to_ingest = [target_file]
else:
    pending_files = [f for f in incoming_files if f not in ingested_files]
    if not pending_files:
        print(f"[IDEMPOTENT SKIP] Seluruh file ({len(incoming_files)} file) sudah pernah di-ingest. 0 baris ditambahkan.")
    elif ingest_mode == "next_one":
        files_to_ingest = pending_files[:1]
    else:
        files_to_ingest = pending_files

print(f"File tersedia di incoming : {incoming_files}")
print(f"File sudah di Bronze      : {sorted(ingested_files)}")
print(f"File akan diproses kali ini: {files_to_ingest}")
print(f"Jumlah baris awal Bronze  : {rows_before:,}")

---
### **4. Ingest Incremental (Append) ke Tabel Delta Bronze**

In [ ]:
run_ts = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
total_added = 0
ingested_details = []

for fname in files_to_ingest:
    fpath = os.path.join(incoming_dir, fname)
    batch_id = f"batch_{os.path.splitext(fname)[0]}_{run_ts}"

    df_batch = (
        spark.read
        .option("header", "true")
        .option("sep", ";")
        .schema(BRONZE_RAW_SCHEMA)
        .csv(fpath)
    )

    df_bronze_batch = (
        df_batch
        .withColumn("batch_id", F.lit(batch_id))
        .withColumn("source_file", F.lit(fname))
        .withColumn("source_path", F.lit(fpath))
        .withColumn("ingested_at", F.current_timestamp())
    )

    batch_count = df_bronze_batch.count()
    (
        df_bronze_batch.write
        .format("delta")
        .mode("append")
        .saveAsTable(full_bronze_table)
    )

    total_added += batch_count
    ingested_details.append({"source_file": fname, "batch_id": batch_id, "rows": batch_count})
    print(f"[APPEND SUKSES] {fname} -> {batch_id} ({batch_count:,} baris)")

rows_after = spark.table(full_bronze_table).count()
print("\n=== RINGKASAN EKSEKUSI BRONZE ===")
print(f"Baris sebelum run : {rows_before:,}")
print(f"Baris ditambahkan : {total_added:,}")
print(f"Baris sesudah run : {rows_after:,}")

---
### **5. Bukti Non-Duplikasi Per Batch & Output Task**

In [ ]:
df_bronze_current = spark.table(full_bronze_table)

print("=== BUKTI AUDIT PER BATCH DI TABEL DELTA BRONZE ===")
batch_summary = (
    df_bronze_current
    .groupBy("source_file", "batch_id")
    .agg(
        F.count("*").alias("jumlah_baris"),
        F.min("ingested_at").alias("waktu_ingest_mulai"),
        F.max("ingested_at").alias("waktu_ingest_selesai"),
    )
    .orderBy("source_file", "waktu_ingest_mulai")
)
display(batch_summary)

print("=== SAMPEL 5 BARIS DATA BRONZE BESERTA METADATA ===")
display(
    df_bronze_current
    .select("age", "job", "balance", "month", "y", "batch_id", "source_file", "ingested_at")
    .limit(5)
)

result_payload = {
    "status": "SUCCESS",
    "table": full_bronze_table,
    "ingest_mode": ingest_mode,
    "files_ingested": files_to_ingest,
    "rows_before": rows_before,
    "rows_added": total_added,
    "rows_after": rows_after,
}
dbutils.notebook.exit(json.dumps(result_payload))